In [1]:
import itertools
def rpnsolve(tokens):
    stack=[]
    for token in tokens:
        if token in "+-*/":
            if len(stack) < 2:
                return None
            b=stack.pop()
            a=stack.pop()
            try: 
                if token=="+":
                    stack.append(a+b)
                elif token=="-":
                    stack.append(a-b)
                elif token=="*":
                    stack.append(a*b)
                elif token=="/":
                    if abs(b)<1e-6:
                        return None

                    stack.append(a/b)
            except Exception:
                return None
        else:
            stack.append(float(token))
    if len(stack) != 1:
        return None
    return stack.pop()

#tokens = input().split()
#print(rpnsolve(tokens))

In [2]:
def tfgenerate(nums):
    ops=["+","-","*","/"]
    expressions=[]

    for prob in itertools.permutations(nums):
        a,b,c,d=prob
        for op1,op2,op3 in itertools.product(ops,repeat=3):
            expressions.append([str(a), str(b), op1, str(c), op2, str(d), op3])
            expressions.append([str(a), str(b), str(c), op2, op1, str(d), op3])
            expressions.append([str(a), str(b), str(c), op2, str(d), op3, op1])
            expressions.append([str(a), str(b), str(c), str(d), op3, op2, op1])
            expressions.append([str(a), str(b), op1, str(c), str(d), op3, op2])

    return expressions
print(tfgenerate([1,2,3,4]))

[['1', '2', '+', '3', '+', '4', '+'], ['1', '2', '3', '+', '+', '4', '+'], ['1', '2', '3', '+', '4', '+', '+'], ['1', '2', '3', '4', '+', '+', '+'], ['1', '2', '+', '3', '4', '+', '+'], ['1', '2', '+', '3', '+', '4', '-'], ['1', '2', '3', '+', '+', '4', '-'], ['1', '2', '3', '+', '4', '-', '+'], ['1', '2', '3', '4', '-', '+', '+'], ['1', '2', '+', '3', '4', '-', '+'], ['1', '2', '+', '3', '+', '4', '*'], ['1', '2', '3', '+', '+', '4', '*'], ['1', '2', '3', '+', '4', '*', '+'], ['1', '2', '3', '4', '*', '+', '+'], ['1', '2', '+', '3', '4', '*', '+'], ['1', '2', '+', '3', '+', '4', '/'], ['1', '2', '3', '+', '+', '4', '/'], ['1', '2', '3', '+', '4', '/', '+'], ['1', '2', '3', '4', '/', '+', '+'], ['1', '2', '+', '3', '4', '/', '+'], ['1', '2', '+', '3', '-', '4', '+'], ['1', '2', '3', '-', '+', '4', '+'], ['1', '2', '3', '-', '4', '+', '+'], ['1', '2', '3', '4', '+', '-', '+'], ['1', '2', '+', '3', '4', '+', '-'], ['1', '2', '+', '3', '-', '4', '-'], ['1', '2', '3', '-', '+', '4', '-'], 

In [3]:
def tfsearch(nums):
    expressions=tfgenerate(nums)
    true_ex=[]
    for expression in expressions:
        res=rpnsolve(expression)
        if res is not None and abs(res-24)<1e-6:
            true_ex.append(expression)

    return true_ex


In [4]:
def rpn2in(tokens):
    stack=[]
    for token in tokens:
        if token in "+-*/":
            a=stack.pop()
            b=stack.pop()
            stack.append("(" + b + token + a + ")")
        else:
            stack.append(token)

    return stack.pop()

In [7]:
nums = [8,3,5,6]
valid_rpn = tfsearch(nums)
if valid_rpn:
    for rpn in valid_rpn:
        infix = rpn2in(rpn)
        print(f"{infix}=24")
else:
    print("No solution found")

(8*(3*(6-5)))=24
((8*3)*(6-5))=24
(8*(3/(6-5)))=24
((8*3)/(6-5))=24
((8/(5-3))*6)=24
(8/((5-3)/6))=24
(8*(5-(6/3)))=24
((8*(6-5))*3)=24
(8*((6-5)*3))=24
(8*(6/(5-3)))=24
((8*6)/(5-3))=24
((8/(6-5))*3)=24
(8/((6-5)/3))=24
(3*(8*(6-5)))=24
((3*8)*(6-5))=24
(3*(8/(6-5)))=24
((3*8)/(6-5))=24
((3*(6-5))*8)=24
(3*((6-5)*8))=24
((3/(6-5))*8)=24
(3/((6-5)/8))=24
((5-(6/3))*8)=24
(6*(8/(5-3)))=24
((6*8)/(5-3))=24
(((6-5)*8)*3)=24
((6-5)*(8*3))=24
(((6-5)*3)*8)=24
((6-5)*(3*8))=24
((6/(5-3))*8)=24
(6/((5-3)/8))=24


In [6]:
import random
import time
def solve_oracle(nums):
    """
    标准的 24 点求解器，用于生成 Ground Truth (真值)
    使用递归 DFS 遍历所有可能的二叉树结构和运算符
    """
    if len(nums) == 1:
        return abs(nums[0] - 24) < 1e-6
    
    for i in range(len(nums)):
        for j in range(len(nums)):
            if i == j:
                continue
            # 剩余的数字
            next_nums = [nums[k] for k in range(len(nums)) if k != i and k != j]
            
            a, b = nums[i], nums[j]
            # 尝试所有运算
            results = [a + b, a - b, b - a, a * b]
            if abs(b) > 1e-6:
                results.append(a / b)
            if abs(a) > 1e-6:
                results.append(b / a)
            
            for res in results:
                if solve_oracle(next_nums + [res]):
                    return True
    return False

def generate_dataset(size):
    """生成随机数据集 (扑克牌点数 1-13)"""
    dataset = []
    for _ in range(size):
        # 随机生成 4 个 1-13 之间的整数
        nums = [random.randint(1, 13) for _ in range(4)]
        dataset.append(nums)
    return dataset

def evaluate_accuracy(dataset):
    """
    评估算法准确度
    指标：
    1. Accuracy (整体准确率): (TP + TN) / Total
    2. Recall (召回率/查全率): TP / (TP + FN) -> 有解的情况下能找到解的比例
    3. Precision (精确率): TP / (TP + FP) -> 找到解的情况下确实是真解的比例
    """
    total = len(dataset)
    tp = 0 # True Positive: 真有解，且算法找到
    tn = 0 # True Negative: 真无解，且算法没找到
    fp = 0 # False Positive: 真无解，但算法找到了 (理论上应为 0)
    fn = 0 # False Negative: 真有解，但算法没找到
    
    start_time = time.time()
    
    for nums in dataset:
        # 1. 获取真值 (Oracle)
        has_solution_truth = solve_oracle([float(n) for n in nums])
        
        # 2. 获取算法结果
        found_expressions = tfsearch(nums)
        has_solution_algo = len(found_expressions) > 0
        
        # 3. 统计
        if has_solution_truth and has_solution_algo:
            tp += 1
        elif not has_solution_truth and not has_solution_algo:
            tn += 1
        elif not has_solution_truth and has_solution_algo:
            fp += 1
        elif has_solution_truth and not has_solution_algo:
            fn += 1
            
    end_time = time.time()
    duration = end_time - start_time
    
    # 计算指标
    accuracy = (tp + tn) / total if total > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    return {
        "total": total,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": accuracy,
        "recall": recall,
        "precision": precision,
        "time": duration
    }

def print_report(size, results):
    print(f"\n{'='*20} 数据集大小：{size} 组 {'='*20}")
    print(f"总用例数：{results['total']}")
    print(f"真有解数 (Oracle): {results['tp'] + results['fn']}")
    print(f"算法找到解数：{results['tp'] + results['fp']}")
    print(f"- True Positive (正确找到): {results['tp']}")
    print(f"- True Negative (正确放弃): {results['tn']}")
    print(f"- False Negative (漏解): {results['fn']}")
    print(f"- False Positive (误报): {results['fp']}")
    print(f"耗时：{results['time']:.4f} 秒")
    print(f"整体准确度 (Accuracy): {results['accuracy']:.2%}")
    print(f"召回率 (Recall): {results['recall']:.2%}  (关键指标：有解时能否找到)")
    print(f"精确率 (Precision): {results['precision']:.2%}")
    print("="*50)

if __name__ == "__main__":
    print("开始 24 点游戏算法准确度测试...")
    print("注意：原代码中的目标值判断已修正为 24，除零错误已处理。")
    
    # 测试用例 1: 100 组数据
    dataset_100 = generate_dataset(100)
    results_100 = evaluate_accuracy(dataset_100)
    print_report(100, results_100)
    
    # 测试用例 2: 10000 组数据
    # 注意：由于原算法遍历量较大，10000 组可能需要较长时间
    print("\n正在生成 10000 组数据集并进行测试，请稍候...")
    dataset_10000 = generate_dataset(10000)
    results_10000 = evaluate_accuracy(dataset_10000)
    print_report(10000, results_10000)
    
    # 示例展示：打印一个具体解的例子
    print("\n示例解展示 (来自 100 组数据中的第一个有解用例):")
    for nums in dataset_100:
        sol = tfsearch(nums)
        if sol:
            infix = rpn2in(sol[0])
            print(f"数字：{nums} -> {infix} = 24")
            break

开始 24 点游戏算法准确度测试...
注意：原代码中的目标值判断已修正为 24，除零错误已处理。

==================== 数据集大小：100 组 ====================
总用例数：100
真有解数 (Oracle): 71
算法找到解数：71
- True Positive (正确找到): 71
- True Negative (正确放弃): 29
- False Negative (漏解): 0
- False Positive (误报): 0
耗时：1.4601 秒
整体准确度 (Accuracy): 100.00%
召回率 (Recall): 100.00%  (关键指标：有解时能否找到)
精确率 (Precision): 100.00%

正在生成 10000 组数据集并进行测试，请稍候...

==================== 数据集大小：10000 组 ====================
总用例数：10000
真有解数 (Oracle): 7888
算法找到解数：7888
- True Positive (正确找到): 7888
- True Negative (正确放弃): 2112
- False Negative (漏解): 0
- False Positive (误报): 0
耗时：269.9424 秒
整体准确度 (Accuracy): 100.00%
召回率 (Recall): 100.00%  (关键指标：有解时能否找到)
精确率 (Precision): 100.00%

示例解展示 (来自 100 组数据中的第一个有解用例):
数字：[12, 3, 2, 10] -> ((12*3)-(2+10)) = 24
